<a href="https://colab.research.google.com/github/zish-s/ML-Zoomcamp-HW/blob/main/homework_03_26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [67]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
)

df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


mode of industry

In [68]:
df.industry.value_counts()

#technology is the mode of industry

,count
industry,
technology,1173
retail,925
healthcare,840
finance,720
education,639
manufacturing,463


Biggest correlation

In [69]:
numerical = df.select_dtypes(include=np.number)


In [70]:
numerical.corr()

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000000,0.137542,0.106375,0.231396,0.181752
number_of_courses_viewed,0.137542,1.000000,0.721609,0.769352,0.395038
interaction_count,0.106375,0.721609,1.000000,0.928712,0.448624
lead_score,0.231396,0.769352,0.928712,1.000000,0.490206
converted,0.181752,0.395038,0.448624,0.490206,1.000000


In [71]:
print(df[['interaction_count', 'lead_score']].corr().iloc[0, 1])
print(df[['number_of_courses_viewed', 'lead_score']].corr().iloc[0, 1])
print(df[['number_of_courses_viewed', 'interaction_count']].corr().iloc[0, 1])
print(df[['annual_income', 'interaction_count']].corr().iloc[0, 1])

0.9287123432397134
0.769352062184664
0.7216090038937035
0.10637458469219016


In [72]:
#the largest correlation quotient - 'interaction_count' and 'lead_score'

Biggest Mutual Information

In [73]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=42
)

df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

In [74]:
y_train = df_train.converted
y_val = df_val.converted

df_train = df_train.drop('converted', axis=1)
df_val = df_val.drop('converted', axis=1)

In [75]:
categorical = df_train.select_dtypes(include='object').columns
numerical = df_train.select_dtypes(exclude='object').columns

df_train[categorical] = df_train[categorical].fillna('NA')
df_val[categorical] = df_val[categorical].fillna('NA')

df_train[numerical] = df_train[numerical].fillna(0.0)
df_val[numerical] = df_val[numerical].fillna(0.0)

In [76]:
from sklearn.metrics import mutual_info_score

for column in categorical:
    score = mutual_info_score(df_train[column], y_train)
    print(column, round(score, 2))

#the most scores one is lead score

lead_source 0.03
industry 0.0
employment_status 0.02
location 0.0


the accuracy

In [77]:
from sklearn.feature_extraction import DictVectorizer

dv = DictVectorizer(sparse=False)

train_dict = df_train.to_dict(orient='records')
val_dict = df_val.to_dict(orient='records')

X_train = dv.fit_transform(train_dict)
X_val = dv.transform(val_dict)

In [78]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    solver='liblinear',
    C=1.0,
    max_iter=1000,
    random_state=42
)

model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [79]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_val)

accuracy = accuracy_score(y_val, y_pred)

print(round(accuracy, 2))

0.65


Feature Selection


In [80]:
original_accuracy = accuracy_score(y_val, y_pred)

print(original_accuracy)

0.645


In [81]:
features_to_remove = [
    'lead_source',
    'number_of_courses_viewed',
    'interaction_count'
]

In [82]:
for feature in features_to_remove:

    train_temp = df_train.drop(feature, axis=1)
    val_temp = df_val.drop(feature, axis=1)

    train_dict = train_temp.to_dict(orient='records')
    val_dict = val_temp.to_dict(orient='records')

    dv_temp = DictVectorizer(sparse=False)

    X_train_temp = dv_temp.fit_transform(train_dict)
    X_val_temp = dv_temp.transform(val_dict)

    model_temp = LogisticRegression(
        solver='liblinear',
        C=1.0,
        max_iter=1000,
        random_state=42
    )

    model_temp.fit(X_train_temp, y_train)

    y_pred_temp = model_temp.predict(X_val_temp)

    accuracy_temp = accuracy_score(y_val, y_pred_temp)

    difference = original_accuracy - accuracy_temp

    print(feature, difference)

lead_source 0.0030000000000000027
number_of_courses_viewed 0.0020000000000000018
interaction_count 0.04400000000000004


Parameter Tuning

In [83]:
C = [0.000001, 0.00001, 0.0001, 0.001]

In [84]:
#train and test

C_values = [0.000001, 0.00001, 0.0001, 0.001]

for C in C_values:

    model = LogisticRegression(
        solver='liblinear',
        C=C,
        max_iter=1000,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)

    accuracy = accuracy_score(y_val, y_pred)

    print(C, round(accuracy, 3))

1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645


In [85]:
print("=== Homework Final Answers ===")
print("1. Mode for industry: technology")
print("2. Biggest correlation: interaction_count and lead_score")
print("3. Biggest MI: lead_source")
print("4. Accuracy: 0.65")
print("5. Feature selection: number_of_courses_viewed")
print("6. Parameter tuning: 0.001")


=== Homework Final Answers ===
1. Mode for industry: technology
2. Biggest correlation: interaction_count and lead_score
3. Biggest MI: lead_source
4. Accuracy: 0.65
5. Feature selection: number_of_courses_viewed
6. Parameter tuning: 0.001
